In [ ]:
import cv2
import numpy as np
from PIL import Image, ImageOps
import os

from breadboard_normalizer.normalizer import Normalizer, PinGrid

normalizer = Normalizer()

image_dir = "example_training_images/"
image_dir = "C:/Users/hartley/Documents/School/spring2026/CS470/stage2/resistor-pose/raw_easy_resistor_images/"
out_dir = "C:/Users/hartley/Documents/School/spring2026/CS470/stage2/resistor-pose/norm_easy_resistor_images/"

window_name = "annotated image"
cv2.namedWindow(window_name, cv2.WINDOW_NORMAL)

for index, file in enumerate(os.listdir(os.fsencode(image_dir))):
    filename, ext = os.path.splitext(os.fsdecode(file))
    if ext.lower().endswith(Normalizer._image_extensions):
        image_pil = Image.open(image_dir + filename + ext)
        image_pil = ImageOps.exif_transpose(image_pil)
        image = np.array(image_pil)

        norm, source_corners, score = normalizer.normalize_image(image, registration="icp_ransac")

        if norm is None:
            continue
        
        point_color = (0, 255, 0)
        if score < 0.5:
            point_color = (0, 0, 255)
            continue
        elif score <= 0.75:
            point_color = (0, 255, 255)

        h = cv2.getPerspectiveTransform(source_corners, normalizer.destination_corners)
        h_inv = np.linalg.inv(h)

        source_points = PinGrid.transform_points_3x3(normalizer.pingrid.points, h_inv)

        if norm is None:
            print("Failed to normalize image")
        else:
            # CV2 assumes BGR
            norm = np.flip(norm, axis=-1)
            image = np.flip(image, axis=-1)

            cv2.imwrite(out_dir + filename + ext, norm)

            # image = draw_corners(image, source_corners)

            for x, y in normalizer.pingrid.points:
                norm = cv2.circle(norm.astype(np.uint8), (int(x), int(y)), 2, point_color, -1)
            
            # for x, y in source_points:
            #     image = cv2.circle(image, (int(x), int(y)), 16, point_color, -1)

            # image = resize_width(image, normalizer.target_size[0])

            # image = np.vstack([image, norm])

            cv2.imshow(window_name, norm)
            if cv2.waitKey(16) == ord('q'):
                break
cv2.destroyAllWindows()